In [1]:
using LatticeDiracOperators
using LatticeQCD
using Gaugefields
using Wilsonloop
using LMP





┌ Warning: Circular dependency detected. Precompilation will be skipped for:
│   SpecialFunctionsExt [1285c0f1-ff9f-5867-b66e-0f359bcf09ba]
│   GPUArrays [0c68f7d7-f131-5f86-a1c3-88cf8149b2d7]
│   LMP [a75e356b-bbea-40ef-9c9d-31e1efba0018]
│   KernelAbstractions [63c18a36-062a-441e-b654-da1e3ab1ce7c]
│   CUDA [052768ef-5323-5732-b1bb-66c8b64840ba]
│   SparseArraysExt [85068d23-b5fb-53f1-8204-05c2aba6942f]
│   AtomixCUDAExt [13011619-4c7c-5ef0-948f-5fc81565cd05]
│   LinearAlgebraExt [66d79d19-2cc4-5b0b-ac7a-b340256d1ecd]
└ @ Pkg.API C:\Users\_s2220186\AppData\Local\Programs\Julia-1.10.7\share\julia\stdlib\v1.10\Pkg\src\API.jl:1279
[ Info: Precompiling LMP [a75e356b-bbea-40ef-9c9d-31e1efba0018]
[ Info: Skipping precompilation since __precompile__(false). Importing LMP [a75e356b-bbea-40ef-9c9d-31e1efba0018].
┌ Warning: Error requiring `Plots` from `LatticeQCD`
│   exception =
│    LoadError: ArgumentError: Package LatticeQCD does not have Plots in its dependencies:
│    - You may have a p

In [2]:
# const pes =  (2, 2, 2, 2)
# const cuda = true
# const numtrj_in = 2 # the number of operations to take average of performance
# const MDsteps_in = 0 # dummy
# PEs = pes

In [3]:
NX = 8
NY = 8
NZ = 8
NT = 16
Nwing = 0
Dim = 4
NC = 3
U = Initialize_4DGaugefields(NC,Nwing,NX,NY,NZ,NT,condition = "hot")
# U = Initialize_Gaugefields(NC,Nwing,NX,NY,NZ,NT,condition = "hot",cuda=true,blocks=PEs)
x = Initialize_pseudofermion_fields(U[1],"Wilson",nowing=true)
gauss_distribution_fermion!(x)
t = Initialize_pseudofermion_fields(U[1],"Wilson",nowing=true)
gauss_distribution_fermion!(t)
x=x/sqrt(norm2(x))
t=t/sqrt(norm2(t))





epsilon=1e-8
P=Onestep()
hop=Hopping_operator(U,P)
W=Generate_weight(hop)






9-element Vector{Spin_matrix}:
 [0.33467955650352993 + 0.0im -0.5271467131933106 + 0.0im 0.844847752498728 + 0.0im 0.4867801086967759 + 0.0im; -0.4204640665444201 + 0.0im -0.8816716912261874 + 0.0im -0.41289998428488084 + 0.0im -1.286591381295191 + 0.0im; 0.4628427399541345 + 0.0im 0.5747897693672354 + 0.0im -1.0805747514630146 + 0.0im 1.2170542114180571 + 0.0im; 0.9341322653545695 + 0.0im -2.063010056826731 + 0.0im -0.3423644660224362 + 0.0im 0.6520357894607544 + 0.0im]
 [-0.7246829881669093 + 0.0im -0.1373984212811607 + 0.0im -1.1618630060531263 + 0.0im 0.44313951628027 + 0.0im; 1.1004121014684332 + 0.0im -0.5490687638904655 + 0.0im -0.04113962047098293 + 0.0im 0.1112614836789849 + 0.0im; 1.262567597318137 + 0.0im 0.49352553477819083 + 0.0im 0.6700473802422288 + 0.0im -0.061761180841145444 + 0.0im; 0.5739610171129076 + 0.0im -0.06365957969008651 + 0.0im 0.08865053503971627 + 0.0im 0.3706017137607827 + 0.0im]
 [-0.20761849551174724 + 0.0im 2.2894591469773835 + 0.0im 1.9183836795886724

In [4]:
m=Matmul!(t,t)

4×4 Spin_matrix:
  0.000675992+0.000481242im  …   -0.00166978+0.000274695im
  0.000396993+0.00167471im        0.00203192-0.00149093im
 -0.000563141-0.000243446im      -0.00132916-0.00258248im
  -0.00166978+0.000274695im     -0.000505353-0.000143212im

In [5]:
layer=CPTC_Layer(hop,W)
act=Softmax()
Attention=Attention_Layer(layer,1,act)
Attention2=Attention_Layer(layer,2,act)
Attention3=Attention_Layer(layer,3,act)
Norm=NormLayer()

# layers=[Attention,Norm,Attention2,Norm,Attention3]
layers=[Attention,Norm,Attention2,Norm,Attention3]
y=layers(x)
f=norm2(y-t)

grad1=deepcopy(Attention.Q.W)
grad2=deepcopy(Attention.K.W)
grad3=deepcopy(Attention.V.W)




9-element Vector{Spin_matrix}:
 [0.5718776192773993 + 0.0im 0.14257153773585746 + 0.0im 0.5852546305986608 + 0.0im 0.1166149779130472 + 0.0im; 0.7114770155202087 + 0.0im 1.7230754743857528 + 0.0im -0.7866135622043612 + 0.0im -1.682359047243244 + 0.0im; -1.3410574909334452 + 0.0im -0.3160177888364651 + 0.0im 0.3153059654548558 + 0.0im 1.8795985836296871 + 0.0im; -0.8540604925211303 + 0.0im -0.7276740356317062 + 0.0im 0.5010855100844805 + 0.0im -0.5942705526342571 + 0.0im]
 [0.3380134762048145 + 0.0im -0.7237542980066074 + 0.0im 1.2672187750605932 + 0.0im 0.05446783269548671 + 0.0im; 1.0316259426585819 + 0.0im -0.6824230103168016 + 0.0im 0.6405478113241915 + 0.0im -0.5661626778475367 + 0.0im; -0.2711214706816097 + 0.0im -0.15898456097655267 + 0.0im 0.9683144535424248 + 0.0im 0.7642796625768752 + 0.0im; -0.7936678212099245 + 0.0im 1.9307374924495175 + 0.0im -0.42472589787914766 + 0.0im 0.604165733976679 + 0.0im]
 [-0.4590296317791267 + 0.0im 1.5647143800251087 + 0.0im -1.3178370577207106 

In [6]:

epsilon=1e-6

sub=layers[2:end]

for i in 1:9
    for u in 1:4
        for v in 1:4
            local layer1=deepcopy(layers[1])
            layer1.Q.W[i][u,v]+=epsilon
            y1=layer1(x)
            grad1[i][u,v]=(norm2(sub(y1)-t)-f)/epsilon
        end
    end
end




for i in 1:9
    for u in 1:4
        for v in 1:4
            local layer1=deepcopy(layers[1])
            layer1.K.W[i][u,v]+=epsilon
            y1=layer1(x)
            grad2[i][u,v]=(norm2(sub(y1)-t)-f)/epsilon
        end
    end
end 

for i in 1:9
    for u in 1:4
        for v in 1:4
            local layer1=deepcopy(layers[1])
            layer1.V.W[i][u,v]+=epsilon
            y1=layer1(x)
            grad3[i][u,v]=(norm2(sub(y1)-t)-f)/epsilon
        end
    end
end 


In [7]:
γ=Clifford()
u = forward_propagation(layers, x,t)
delta = backward_propagation(layers, t, u)

d = gradient(layers[1], layers[1].activation, u[1], delta[5],γ)








3-element Vector{Vector{Spin_matrix}}:
 [[-0.10156454791254874 + 0.0im -0.15252308581164845 + 0.0im -0.19432945049603834 + 0.0im 0.06313816894912061 + 0.0im; 0.7129842571294163 + 0.0im -0.006472354443089023 + 0.0im 0.3473819709904255 + 0.0im -0.41847186046598195 + 0.0im; -0.24533012328613313 + 0.0im -0.48477803550960297 + 0.0im -0.19115159744457594 + 0.0im -0.46071625252327886 + 0.0im; -0.05118620570488308 + 0.0im -0.10748985710297607 + 0.0im 0.15491915077615726 + 0.0im 0.6832342352839766 + 0.0im], [0.005502954485734568 + 0.0im 0.23303050876498274 + 0.0im -0.005014084176913902 + 0.0im -0.387934340281526 + 0.0im; 0.3735141935260289 + 0.0im 0.3244507810972331 + 0.0im 0.11741753738568783 + 0.0im 0.3538738825026596 + 0.0im; -0.0777453531472238 + 0.0im 0.2033968819548128 + 0.0im 0.02372069096182436 + 0.0im -0.07547279905391442 + 0.0im; 0.6079359424356826 + 0.0im 0.22239815846452607 + 0.0im -0.4161562977786627 + 0.0im 0.08376588469093078 + 0.0im], [-0.15815022604579942 + 0.0im -0.49188623826

In [8]:
d

3-element Vector{Vector{Spin_matrix}}:
 [[-0.10156454791254874 + 0.0im -0.15252308581164845 + 0.0im -0.19432945049603834 + 0.0im 0.06313816894912061 + 0.0im; 0.7129842571294163 + 0.0im -0.006472354443089023 + 0.0im 0.3473819709904255 + 0.0im -0.41847186046598195 + 0.0im; -0.24533012328613313 + 0.0im -0.48477803550960297 + 0.0im -0.19115159744457594 + 0.0im -0.46071625252327886 + 0.0im; -0.05118620570488308 + 0.0im -0.10748985710297607 + 0.0im 0.15491915077615726 + 0.0im 0.6832342352839766 + 0.0im], [0.005502954485734568 + 0.0im 0.23303050876498274 + 0.0im -0.005014084176913902 + 0.0im -0.387934340281526 + 0.0im; 0.3735141935260289 + 0.0im 0.3244507810972331 + 0.0im 0.11741753738568783 + 0.0im 0.3538738825026596 + 0.0im; -0.0777453531472238 + 0.0im 0.2033968819548128 + 0.0im 0.02372069096182436 + 0.0im -0.07547279905391442 + 0.0im; 0.6079359424356826 + 0.0im 0.22239815846452607 + 0.0im -0.4161562977786627 + 0.0im 0.08376588469093078 + 0.0im], [-0.15815022604579942 + 0.0im -0.49188623826

In [9]:
grad1

9-element Vector{Spin_matrix}:
 [-0.10156334440125647 + 0.0im -0.15252218332761913 + 0.0im -0.19432831521726257 + 0.0im 0.06313902645160852 + 0.0im; 0.7129852264142755 + 0.0im -0.006471346125636046 + 0.0im 0.3473834055967018 + 0.0im -0.4184715436394981 + 0.0im; -0.24532965170465104 + 0.0im -0.4847762653525933 + 0.0im -0.19115065441610568 + 0.0im -0.46071666304214887 + 0.0im; -0.051185757854455005 + 0.0im -0.10748902212753819 + 0.0im 0.15492135929662254 + 0.0im 0.6832358607766764 + 0.0im]
 [0.005504404043676914 + 0.0im 0.2330329067490311 + 0.0im -0.005012566361983772 + 0.0im -0.38793353240862416 + 0.0im; 0.37351492210291326 + 0.0im 0.32445187514440477 + 0.0im 0.11741840921786206 + 0.0im 0.35387567720590596 + 0.0im; -0.07774443133712339 + 0.0im 0.20339755479881205 + 0.0im 0.023721050013136846 + 0.0im -0.07547146907427305 + 0.0im; 0.6079382615098439 + 0.0im 0.2223997839223557 + 0.0im -0.4161557640713909 + 0.0im 0.08376597548931386 + 0.0im]
 [-0.15814850229389776 + 0.0im -0.491886389397677

In [10]:
println(grad1[2]-d[1][2])
println(grad2[2]-d[2][2])
println(grad3[2]-d[3][2])
    


ComplexF64[1.4495579423456917e-6 + 0.0im 2.397984048357893e-6 + 0.0im 1.5178149301298127e-6 + 0.0im 8.07872901853468e-7 + 0.0im; 7.285768843789953e-7 + 0.0im 1.09404717169026e-6 + 0.0im 8.718321742318125e-7 + 0.0im 1.7947032463494494e-6 + 0.0im; 9.218101004182522e-7 + 0.0im 6.728439992387258e-7 + 0.0im 3.5905131248736843e-7 + 0.0im 1.3299796413762666e-6 + 0.0im; 2.319074161261092e-6 + 0.0im 1.6254578296404087e-6 + 0.0im 5.337072718214486e-7 + 0.0im 9.079838307612942e-8 + 0.0im]
ComplexF64[-7.807501693829141e-7 + 0.0im 9.199764331585536e-7 + 0.0im 1.7794752602828812e-6 + 0.0im 4.210597541465244e-7 + 0.0im; 9.878124124715337e-7 + 0.0im 1.437710035231965e-6 + 0.0im 7.437273444044212e-7 + 0.0im 8.008705152107165e-7 + 0.0im; 1.19099727656502e-6 + 0.0im 1.964852240132764e-6 + 0.0im 1.0178523086878943e-6 + 0.0im 1.6139954954474955e-6 + 0.0im; 4.0287415076223354e-7 + 0.0im 1.0084488815884085e-6 + 0.0im 1.3624358734254383e-6 + 0.0im 3.467882898414154e-7 + 0.0im]
ComplexF64[6.341129648868016e-7 

In [14]:
println(grad3[2])
println(d[3][2])

ComplexF64[-0.14709501527931934 + 0.0im 0.12454994191557489 + 0.0im -0.14900960110253436 + 0.0im 0.09792482558168558 + 0.0im; -0.01943939054172006 + 0.0im 0.17171685584571605 + 0.0im -0.03012057447904226 + 0.0im 0.11136692990021402 + 0.0im; -0.2721030369912114 + 0.0im -0.08031730658331071 + 0.0im 0.028992879208544764 + 0.0im -0.10996678412311667 + 0.0im; -0.1542396290687975 + 0.0im 0.03291316374998132 + 0.0im 0.11256386756031134 + 0.0im -0.01133623328541944 + 0.0im]
ComplexF64[-0.14709564939228423 + 0.0im 0.1245504584598136 + 0.0im -0.1490109116994067 + 0.0im 0.09792493546828902 + 0.0im; -0.019439979032391183 + 0.0im 0.17171601858584373 + 0.0im -0.03012134230961911 + 0.0im 0.11136493336348108 + 0.0im; -0.2721027745180701 + 0.0im -0.08031755951862801 + 0.0im 0.028992177802969 + 0.0im -0.10996692506622298 + 0.0im; -0.1542403138766644 + 0.0im 0.03291177475784513 + 0.0im 0.11256306044698476 + 0.0im -0.011335899521415161 + 0.0im]
